# Train Full-Precision CNN and RCNN Decoders

This notebook trains full-precision (32-bit) CNN and RCNN models across all 12 (d, p) configs.
Results are saved to `results_fullprecision.csv` and serve as:
- Experiment 1 comparison (CNN vs RCNN vs MWPM)
- 32-bit baseline for quantization sweep


In [4]:
import numpy as np
import tensorflow as tf
import os
import csv
from datetime import datetime

In [5]:
print(f'Starting full-precision training at {datetime.now()}')
print(f'TensorFlow version: {tf.__version__}')

# Configuration
CONFIGS = [
    (3, 0.001), (3, 0.005), (3, 0.010), (3, 0.050),
    (5, 0.001), (5, 0.005), (5, 0.010), (5, 0.050),
    (7, 0.001), (7, 0.005), (7, 0.010), (7, 0.050),
]
ROUNDS = 2
DATA_DIR = './datasets'
OUTPUT_DIR = './results'
SEED = 42

# Training hyperparameters
BATCH_SIZE = 256
LEARNING_RATE = 1e-3
PATIENCE = 10
MAX_EPOCHS = 100

os.makedirs(OUTPUT_DIR, exist_ok=True)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print(f'Data dir: {DATA_DIR}')
print(f'Output dir: {OUTPUT_DIR}\n')

Starting full-precision training at 2026-06-06 13:28:10.819511
TensorFlow version: 2.15.0
Data dir: ./datasets
Output dir: ./results



In [6]:
def get_2d_shape(n):
    """
    Find best (H, W) rectangle for n features.
    Fixes the int(sqrt(n)) bug: sqrt(48)=6.9 → 6x6=36≠48.
    Examples: n=16→(4,4), n=48→(6,8), n=96→(8,12)
    """
    side = int(np.sqrt(n))
    if side * side == n:
        return side, side
    for h in range(side, 0, -1):
        if n % h == 0:
            return h, n // h
    return 1, n

def build_cnn(input_shape, output_shape):
    """
    Full-precision CNN decoder.
    Input is reshaped to 2D spatial grid using get_2d_shape().
    Conv2D layers extract local syndrome patterns.
    """
    H, W = get_2d_shape(input_shape[0])
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=input_shape),
        tf.keras.layers.Reshape((H, W, 1)),
        tf.keras.layers.Conv2D(32, 3, padding='same', activation='relu'),
        tf.keras.layers.Conv2D(32, 3, padding='same', activation='relu'),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(64, activation='relu'),
        tf.keras.layers.Dense(output_shape[0], activation='sigmoid')
    ])
    return model

def build_rcnn(input_shape, output_shape):
    """
    Full-precision RCNN decoder.
    Conv2D extracts spatial features, LSTM captures temporal correlations
    across syndrome rounds.
    """
    H, W = get_2d_shape(input_shape[0])
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=input_shape),
        tf.keras.layers.Reshape((H, W, 1)),
        tf.keras.layers.Conv2D(32, 3, padding='same', activation='relu'),
        tf.keras.layers.Flatten(),
        tf.keras.layers.RepeatVector(1),
        tf.keras.layers.LSTM(32, activation='relu'),
        tf.keras.layers.Dense(64, activation='relu'),
        tf.keras.layers.Dense(output_shape[0], activation='sigmoid')
    ])
    return model

# Sanity check reshape for all distances
for d in [3, 5, 7]:
    n = (d**2 - 1) * ROUNDS
    H, W = get_2d_shape(n)
    assert H * W == n, f'd={d}: {H}x{W}={H*W} != {n}'
    print(f'd={d}: {n} features -> ({H},{W},1)')
print('CNN and RCNN architectures defined')

d=3: 16 features -> (4,4,1)
d=5: 48 features -> (6,8,1)
d=7: 96 features -> (8,12,1)
CNN and RCNN architectures defined


In [7]:
# CSV path — write header first, then append row by row
# This means results survive interruption (unlike writerows at the end)
csv_path = f'{OUTPUT_DIR}/results_fullprecision.csv'
fieldnames = ['distance', 'noise', 'decoder', 'p_L', 'model_size_kb']

with open(csv_path, 'w', newline='') as f:
    csv.DictWriter(f, fieldnames=fieldnames).writeheader()

results = []

for config_idx, (d, p) in enumerate(CONFIGS, 1):
    print(f'\n[{config_idx}/{len(CONFIGS)}] d={d}, p={p:.3f}')
    print('='*60)

    try:
        # Load dataset
        filename = f'{DATA_DIR}/data_d{d}_p{p:.3f}_r{ROUNDS}.npz'
        data = np.load(filename)
        det_evts = data['det_evts'].astype(np.float32)
        flips = data['flips'].astype(np.float32)

        # Split: 800k train, 100k val, 100k test
        n_train = 800_000
        n_val   = 100_000
        X_train = det_evts[:n_train]
        y_train = flips[:n_train]
        X_val   = det_evts[n_train:n_train+n_val]
        y_val   = flips[n_train:n_train+n_val]
        X_test  = det_evts[n_train+n_val:]
        y_test  = flips[n_train+n_val:]

        input_shape  = X_train.shape[1:]
        output_shape = y_train.shape[1:]

        print(f'Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}\n')

        # --- Train CNN ---
        print('Training CNN...')
        cnn = build_cnn(input_shape, output_shape)
        cnn.compile(
            optimizer=tf.keras.optimizers.legacy.Adam(learning_rate=LEARNING_RATE),
            loss='binary_crossentropy',
            metrics=['accuracy']
        )
        cnn_history = cnn.fit(
            X_train, y_train,
            batch_size=BATCH_SIZE,
            validation_data=(X_val, y_val),
            epochs=MAX_EPOCHS,
            callbacks=[tf.keras.callbacks.EarlyStopping(
                patience=PATIENCE, restore_best_weights=True)],
            verbose=1
        )

        cnn_pred   = (cnn.predict(X_test, verbose=0) > 0.5).astype(float)
        cnn_p_L    = float((cnn_pred != y_test).any(axis=1).sum()) / len(y_test)
        cnn_size   = cnn.count_params() * 4 / 1024
        cnn_epochs = len(cnn_history.history['loss'])
        print(f'CNN p_L={cnn_p_L:.6f} (trained {cnn_epochs} epochs, size={cnn_size:.1f} KB)\n')

        cnn_row = {'distance': d, 'noise': p, 'decoder': 'CNN-32bit',
                   'p_L': round(cnn_p_L, 8), 'model_size_kb': round(cnn_size, 2)}
        results.append(cnn_row)
        with open(csv_path, 'a', newline='') as f:
            csv.DictWriter(f, fieldnames=fieldnames).writerow(cnn_row)

        del cnn
        tf.keras.backend.clear_session()

        # --- Train RCNN ---
        print('Training RCNN...')
        rcnn = build_rcnn(input_shape, output_shape)
        rcnn.compile(
            optimizer=tf.keras.optimizers.legacy.Adam(learning_rate=LEARNING_RATE),
            loss='binary_crossentropy',
            metrics=['accuracy']
        )
        rcnn_history = rcnn.fit(
            X_train, y_train,
            batch_size=BATCH_SIZE,
            validation_data=(X_val, y_val),
            epochs=MAX_EPOCHS,
            callbacks=[tf.keras.callbacks.EarlyStopping(
                patience=PATIENCE, restore_best_weights=True)],
            verbose=1
        )

        rcnn_pred   = (rcnn.predict(X_test, verbose=0) > 0.5).astype(float)
        rcnn_p_L    = float((rcnn_pred != y_test).any(axis=1).sum()) / len(y_test)
        rcnn_size   = rcnn.count_params() * 4 / 1024
        rcnn_epochs = len(rcnn_history.history['loss'])
        print(f'RCNN p_L={rcnn_p_L:.6f} (trained {rcnn_epochs} epochs, size={rcnn_size:.1f} KB)\n')

        rcnn_row = {'distance': d, 'noise': p, 'decoder': 'RCNN-32bit',
                    'p_L': round(rcnn_p_L, 8), 'model_size_kb': round(rcnn_size, 2)}
        results.append(rcnn_row)
        with open(csv_path, 'a', newline='') as f:
            csv.DictWriter(f, fieldnames=fieldnames).writerow(rcnn_row)

        del rcnn
        tf.keras.backend.clear_session()

    except Exception as e:
        print(f'Error at d={d} p={p}: {e}')
        continue

print(f'\nDone: {len(results)}/{len(CONFIGS)*2} runs completed')
print(f'Results saved to {csv_path}')
print(f'Completed at {datetime.now()}')


[1/12] d=3, p=0.001
Train: (800000, 16), Val: (100000, 16), Test: (100000, 16)

Training CNN...
Epoch 1/100
3125/3125 [==============================] - 7s 2ms/step - loss: 0.0100 - accuracy: 0.9982 - val_loss: 0.0029 - val_accuracy: 0.9994
Epoch 2/100
3125/3125 [==============================] - 8s 2ms/step - loss: 0.0025 - accuracy: 0.9994 - val_loss: 0.0026 - val_accuracy: 0.9995
Epoch 3/100
3125/3125 [==============================] - 8s 2ms/step - loss: 0.0022 - accuracy: 0.9995 - val_loss: 0.0024 - val_accuracy: 0.9994
Epoch 4/100
3125/3125 [==============================] - 8s 2ms/step - loss: 0.0020 - accuracy: 0.9995 - val_loss: 0.0022 - val_accuracy: 0.9995
Epoch 5/100
3125/3125 [==============================] - 8s 2ms/step - loss: 0.0019 - accuracy: 0.9995 - val_loss: 0.0023 - val_accuracy: 0.9995
Epoch 6/100
3125/3125 [==============================] - 8s 2ms/step - loss: 0.0018 - accuracy: 0.9996 - val_loss: 0.0021 - val_accuracy: 0.9995
Epoch 7/100
3125/3125 [==========